# 🏭 RL Training for Multi-Supplier Perishable Inventory MDP

This notebook trains a reinforcement learning agent using **PPO (Proximal Policy Optimization)** to manage a perishable inventory system with multiple suppliers.

## Problem Overview

The agent must learn to:
- **Order from suppliers** with different lead times and costs
- **Manage perishable inventory** (items expire after N periods)
- **Balance costs**: purchase, holding, shortage, and spoilage
- **Meet stochastic demand** while minimizing total cost

## MDP Structure

- **State**: Inventory by expiry bucket, supplier pipelines, backorders
- **Action**: Order quantities for each supplier
- **Reward**: Negative of period costs
- **Dynamics**: FIFO consumption, aging, stochastic Poisson demand

## 1. Setup & Installation

In [ ]:
# Install required packages (run this cell first on Colab)
!pip install stable-baselines3[extra] gymnasium numpy scipy matplotlib tensorboard -q

In [ ]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the repository if on Colab
    !git clone https://github.com/your-username/Multi-Supplier-Perishable-Inventory.git repo
    %cd repo
else:
    # Local development - add parent directory to path
    import os
    os.chdir('..')
    
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Optional, Tuple
import time
import os

# RL imports
import gymnasium as gym
from stable_baselines3 import PPO, A2C, SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor

# Inventory simulation imports
from inventory_sim.gym_env import PerishableInventoryGymEnv, make_inventory_env
from inventory_sim.env import create_simple_mdp
from inventory_sim.simulation import run_episode
from inventory_agents import TailoredBaseSurgePolicy, BaseStockPolicy, DoNothingPolicy

print("✅ All imports successful!")

## 2. Environment Configuration

In [ ]:
# Environment hyperparameters
ENV_CONFIG = {
    "shelf_life": 5,              # Items expire after 5 periods
    "num_suppliers": 2,           # Two suppliers
    "mean_demand": 10.0,          # Average demand per period
    "fast_lead_time": 1,          # Fast supplier: 1 period lead time
    "slow_lead_time": 3,          # Slow supplier: 3 period lead time
    "fast_cost": 2.0,             # Fast supplier cost per unit
    "slow_cost": 1.0,             # Slow supplier cost per unit
    "max_order_per_supplier": 30, # Maximum order per supplier
    "order_step": 5,              # Action discretization step
    "max_episode_steps": 200,     # Episode length
    "normalize_obs": True,        # Normalize observations to [0, 1]
}

print("Environment Configuration:")
for key, value in ENV_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Create and test the environment
env = PerishableInventoryGymEnv(**ENV_CONFIG, render_mode="ansi")

print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")
print(f"\nAction meanings:")
print(f"  Each dimension: order quantity = action * {ENV_CONFIG['order_step']}")
print(f"  E.g., action [2, 4] = order {2*5}=10 units from fast, {4*5}=20 units from slow")

In [ ]:
# Test environment with random actions
print("Testing environment with random actions...\n")

obs, info = env.reset(seed=42)
print(f"Initial observation shape: {obs.shape}")
print(f"Initial observation: {obs}")

total_reward = 0
for step in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(f"\nStep {step + 1}:")
    print(f"  Action: {action} (orders: {action * ENV_CONFIG['order_step']})")
    print(f"  Reward: {reward:.2f}")
    print(f"  Demand: {info.get('demand', 'N/A')}, Sales: {info.get('sales', 'N/A')}")
    print(f"  Inventory Position: {info.get('inventory_position', 'N/A'):.0f}")

print(f"\nTotal reward over 5 steps: {total_reward:.2f}")

## 3. Training Configuration

In [ ]:
# Training hyperparameters
TRAINING_CONFIG = {
    "total_timesteps": 100_000,    # Total training steps (increase for better results)
    "n_envs": 4,                   # Number of parallel environments
    "learning_rate": 3e-4,         # Learning rate
    "n_steps": 2048,               # Steps per update
    "batch_size": 64,              # Minibatch size
    "n_epochs": 10,                # Number of epochs per update
    "gamma": 0.99,                 # Discount factor
    "ent_coef": 0.01,              # Entropy coefficient
    "clip_range": 0.2,             # PPO clip range
    "verbose": 1,                  # Verbosity level
}

# Create output directories
os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Create vectorized environment for parallel training
def make_env():
    return PerishableInventoryGymEnv(**ENV_CONFIG)

# Create parallel training environments
train_env = make_vec_env(make_env, n_envs=TRAINING_CONFIG["n_envs"])

# Create evaluation environment
eval_env = Monitor(PerishableInventoryGymEnv(**ENV_CONFIG))

print(f"Created {TRAINING_CONFIG['n_envs']} parallel training environments")

## 4. Create PPO Agent

In [ ]:
# Create PPO model with MLP policy
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=TRAINING_CONFIG["learning_rate"],
    n_steps=TRAINING_CONFIG["n_steps"],
    batch_size=TRAINING_CONFIG["batch_size"],
    n_epochs=TRAINING_CONFIG["n_epochs"],
    gamma=TRAINING_CONFIG["gamma"],
    ent_coef=TRAINING_CONFIG["ent_coef"],
    clip_range=TRAINING_CONFIG["clip_range"],
    verbose=TRAINING_CONFIG["verbose"],
    tensorboard_log="./logs/",
    policy_kwargs={
        "net_arch": [dict(pi=[128, 128], vf=[128, 128])]
    }
)

print(f"Model created: {model.policy}")
print(f"\nPolicy Network Architecture:")
print(model.policy)

## 5. Training

In [ ]:
# Create callbacks for evaluation and checkpointing
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./models/best/",
    log_path="./logs/eval/",
    eval_freq=5000,
    n_eval_episodes=5,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=10000,
    save_path="./models/checkpoints/",
    name_prefix="ppo_inventory"
)

print("Callbacks created for evaluation and checkpointing")

In [ ]:
# Train the model!
print("🚀 Starting training...")
print(f"Total timesteps: {TRAINING_CONFIG['total_timesteps']:,}")
print("-" * 50)

start_time = time.time()

model.learn(
    total_timesteps=TRAINING_CONFIG["total_timesteps"],
    callback=[eval_callback, checkpoint_callback],
    progress_bar=True
)

training_time = time.time() - start_time
print("-" * 50)
print(f"✅ Training completed in {training_time:.1f} seconds ({training_time/60:.1f} minutes)")

In [ ]:
# Save the final model
model.save("models/ppo_inventory_final")
print("Model saved to models/ppo_inventory_final.zip")

## 6. Evaluate Trained Agent

In [ ]:
# Load best model
best_model_path = "models/best/best_model.zip"
if os.path.exists(best_model_path):
    model = PPO.load(best_model_path)
    print("Loaded best model from evaluation callbacks")
else:
    print("Using final trained model")

# Evaluate the trained agent
n_eval_episodes = 10
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=n_eval_episodes)
print(f"\n📊 RL Agent Performance ({n_eval_episodes} episodes):")
print(f"  Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"  Mean Cost per Episode: {-mean_reward:.2f}")

## 7. Compare with Baseline Policies

In [ ]:
# Define baseline policies to compare against
baselines = {
    "Do Nothing": DoNothingPolicy(),
    "Base Stock (S=60)": BaseStockPolicy(target_level=60.0, supplier_id=1),
    "TBS (Base=50, Reorder=25)": TailoredBaseSurgePolicy(
        slow_supplier_id=1,
        fast_supplier_id=0,
        base_stock_level=50.0,
        reorder_point=25.0
    ),
}

# Create MDP for baseline evaluation
mdp = create_simple_mdp(
    shelf_life=ENV_CONFIG["shelf_life"],
    num_suppliers=ENV_CONFIG["num_suppliers"],
    mean_demand=ENV_CONFIG["mean_demand"],
    fast_lead_time=ENV_CONFIG["fast_lead_time"],
    slow_lead_time=ENV_CONFIG["slow_lead_time"],
    fast_cost=ENV_CONFIG["fast_cost"],
    slow_cost=ENV_CONFIG["slow_cost"],
)

In [ ]:
def evaluate_baseline_policy(policy, mdp, n_episodes=10, n_periods=200, seed=42):
    """Evaluate a baseline policy and return metrics."""
    all_rewards = []
    all_fill_rates = []
    all_spoilage_rates = []
    
    for ep in range(n_episodes):
        initial_inv = np.full(mdp.shelf_life, 20.0)
        state = mdp.create_initial_state(initial_inventory=initial_inv)
        results, total_reward = run_episode(
            mdp, policy, num_periods=n_periods, seed=seed + ep, initial_state=state
        )
        metrics = mdp.compute_inventory_metrics(results)
        
        all_rewards.append(total_reward)
        all_fill_rates.append(metrics["fill_rate"])
        all_spoilage_rates.append(metrics["spoilage_rate"])
    
    return {
        "mean_reward": np.mean(all_rewards),
        "std_reward": np.std(all_rewards),
        "mean_fill_rate": np.mean(all_fill_rates),
        "mean_spoilage_rate": np.mean(all_spoilage_rates),
    }

In [ ]:
# Evaluate all baselines
print("Evaluating baseline policies...\n")
baseline_results = {}

for name, policy in baselines.items():
    results = evaluate_baseline_policy(policy, mdp, n_episodes=10, n_periods=200)
    baseline_results[name] = results
    print(f"{name}:")
    print(f"  Mean Reward: {results['mean_reward']:.2f} ± {results['std_reward']:.2f}")
    print(f"  Fill Rate: {results['mean_fill_rate']:.2%}")
    print(f"  Spoilage Rate: {results['mean_spoilage_rate']:.2%}")
    print()

In [ ]:
# Evaluate RL agent on the same setup
print("Evaluating RL agent...\n")

rl_rewards = []
rl_fill_rates = []
rl_spoilage_rates = []

for ep in range(10):
    obs, info = eval_env.reset(seed=42 + ep)
    episode_reward = 0
    total_demand = 0
    total_sales = 0
    total_spoiled = 0
    total_arrivals = 0
    
    for _ in range(200):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        episode_reward += reward
        
        total_demand += info.get('demand', 0)
        total_sales += info.get('sales', 0)
        total_spoiled += info.get('spoiled', 0)
        
        if terminated or truncated:
            break
    
    rl_rewards.append(episode_reward)
    if total_demand > 0:
        rl_fill_rates.append(total_sales / total_demand)
    if total_arrivals > 0:
        rl_spoilage_rates.append(total_spoiled / total_arrivals)

rl_results = {
    "mean_reward": np.mean(rl_rewards),
    "std_reward": np.std(rl_rewards),
    "mean_fill_rate": np.mean(rl_fill_rates) if rl_fill_rates else 0,
    "mean_spoilage_rate": np.mean(rl_spoilage_rates) if rl_spoilage_rates else 0,
}

print(f"RL Agent:")
print(f"  Mean Reward: {rl_results['mean_reward']:.2f} ± {rl_results['std_reward']:.2f}")
print(f"  Fill Rate: {rl_results['mean_fill_rate']:.2%}")
print(f"  Spoilage Rate: {rl_results['mean_spoilage_rate']:.2%}")

## 8. Visualization

In [ ]:
# Compare all policies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prepare data
all_policies = list(baselines.keys()) + ["RL Agent (PPO)"]
all_results = list(baseline_results.values()) + [rl_results]

rewards = [r["mean_reward"] for r in all_results]
reward_stds = [r["std_reward"] for r in all_results]

# Plot 1: Rewards comparison
ax1 = axes[0]
colors = ['gray', 'steelblue', 'forestgreen', 'crimson']
bars = ax1.barh(all_policies, rewards, xerr=reward_stds, color=colors, alpha=0.8)
ax1.set_xlabel("Mean Episode Reward (higher is better)")
ax1.set_title("Policy Comparison: Rewards")
ax1.axvline(x=0, color='black', linestyle='--', alpha=0.3)

# Add value labels
for bar, reward in zip(bars, rewards):
    ax1.text(reward - 50, bar.get_y() + bar.get_height()/2, 
             f'{reward:.0f}', va='center', ha='right', fontweight='bold', color='white')

# Plot 2: Fill rates
ax2 = axes[1]
fill_rates = [r["mean_fill_rate"] * 100 for r in all_results]
spoilage_rates = [r.get("mean_spoilage_rate", 0) * 100 for r in all_results]

x = np.arange(len(all_policies))
width = 0.35

bars1 = ax2.bar(x - width/2, fill_rates, width, label='Fill Rate %', color='forestgreen', alpha=0.8)
bars2 = ax2.bar(x + width/2, spoilage_rates, width, label='Spoilage Rate %', color='crimson', alpha=0.8)

ax2.set_ylabel('Percentage')
ax2.set_title('Policy Comparison: Service Metrics')
ax2.set_xticks(x)
ax2.set_xticklabels(all_policies, rotation=15, ha='right')
ax2.legend()
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('policy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Comparison plot saved to policy_comparison.png")

In [ ]:
# Visualize a single episode with the RL agent
print("Running a sample episode with the trained RL agent...\n")

obs, info = eval_env.reset(seed=123)
episode_data = {
    'step': [],
    'inventory_position': [],
    'demand': [],
    'sales': [],
    'order_fast': [],
    'order_slow': [],
    'reward': [],
}

for step in range(50):  # First 50 steps
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    
    episode_data['step'].append(step)
    episode_data['inventory_position'].append(info.get('inventory_position', 0))
    episode_data['demand'].append(info.get('demand', 0))
    episode_data['sales'].append(info.get('sales', 0))
    episode_data['order_fast'].append(action[0] * ENV_CONFIG['order_step'])
    episode_data['order_slow'].append(action[1] * ENV_CONFIG['order_step'])
    episode_data['reward'].append(reward)
    
    if terminated or truncated:
        break

In [ ]:
# Plot episode trace
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Inventory and Demand
ax1 = axes[0]
ax1.plot(episode_data['step'], episode_data['inventory_position'], 
         'b-', linewidth=2, label='Inventory Position')
ax1.fill_between(episode_data['step'], 0, episode_data['demand'], 
                 alpha=0.3, color='red', label='Demand')
ax1.axhline(y=25, color='orange', linestyle='--', alpha=0.7, label='Reorder Point (ref)')
ax1.axhline(y=50, color='green', linestyle='--', alpha=0.7, label='Base Stock (ref)')
ax1.set_ylabel('Units')
ax1.set_title('RL Agent Episode: Inventory Position and Demand')
ax1.legend(loc='upper right')
ax1.grid(alpha=0.3)

# Plot 2: Orders
ax2 = axes[1]
ax2.bar(episode_data['step'], episode_data['order_fast'], 
        alpha=0.7, label='Fast Supplier Order', color='crimson', width=0.8)
ax2.bar(episode_data['step'], episode_data['order_slow'], 
        alpha=0.7, label='Slow Supplier Order', color='steelblue', width=0.8, bottom=episode_data['order_fast'])
ax2.set_ylabel('Order Quantity')
ax2.set_title('RL Agent Episode: Order Decisions')
ax2.legend(loc='upper right')
ax2.grid(alpha=0.3)

# Plot 3: Cumulative Reward
ax3 = axes[2]
cumulative_reward = np.cumsum(episode_data['reward'])
ax3.plot(episode_data['step'], cumulative_reward, 'g-', linewidth=2)
ax3.fill_between(episode_data['step'], 0, cumulative_reward, alpha=0.3, color='green')
ax3.set_xlabel('Step')
ax3.set_ylabel('Cumulative Reward')
ax3.set_title('RL Agent Episode: Cumulative Reward')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rl_agent_episode.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Episode trace saved to rl_agent_episode.png")

## 9. Save & Export

In [ ]:
# Save the final trained model
model.save("models/ppo_inventory_final")
print("✅ Final model saved to models/ppo_inventory_final.zip")

# If on Colab, download the model
if IN_COLAB:
    from google.colab import files
    files.download('models/ppo_inventory_final.zip')
    print("\n📥 Model download initiated")

In [ ]:
# Summary
print("="*60)
print("🏁 TRAINING COMPLETE")
print("="*60)
print(f"\n📦 Model saved to: models/ppo_inventory_final.zip")
print(f"\n📊 RL Agent vs Best Baseline (TBS):")
print(f"   RL Agent Reward:  {rl_results['mean_reward']:.2f}")
print(f"   TBS Reward:       {baseline_results['TBS (Base=50, Reorder=25)']['mean_reward']:.2f}")

improvement = rl_results['mean_reward'] - baseline_results['TBS (Base=50, Reorder=25)']['mean_reward']
if improvement > 0:
    print(f"   \n✨ RL Agent outperforms TBS by {improvement:.2f} reward units!")
else:
    print(f"   \n📈 Consider training for more steps to improve performance.")
    print(f"   Current gap: {-improvement:.2f} reward units behind TBS.")

## 10. Next Steps

To improve the RL agent's performance:

1. **Train longer**: Increase `total_timesteps` to 500K or 1M
2. **Tune hyperparameters**: Adjust learning rate, network size, etc.
3. **Reward shaping**: Add bonuses for maintaining good inventory position
4. **Try different algorithms**: A2C, SAC (for continuous actions)
5. **Curriculum learning**: Start with easier scenarios

To use the trained model:
```python
from stable_baselines3 import PPO
model = PPO.load("models/ppo_inventory_final")
action, _ = model.predict(observation, deterministic=True)
```